# 19b — Real-data μ identifiability: autopsy + μ-score redesign

**Series 19 — autonomous improvement round (goal: honest uncertainties on the real dataset).**
Trial b: on the REAL data, find where the half-μ attractor comes from (count channel vs
σ_fit channel) and test μ-score designs that could recover μ with honest uncertainty.

**Key insight:** <TBD — filled after the run>


## 1. CURRENT STATE ANALYSIS (written before this trial)

### 1.1 The real-data μ collapse (18-series)
- μ̂ lands at **~½ of the reference** in every real-data experiment, but within huge σ_μ
  (|Δμ|/σ_μ 0.02–1.7, RMSE 46 vs 47): the μ likelihood is FLAT — honest but useless.
- γ is the better-behaved channel: 3nW high-T good, 1nW systematically low (model
  mismatch, failure mode #6), and the z-form (17g/18c) fixed the optimizer-level jumps.

### 1.2 19a killed the line-shape explanation
- Voigt targets (σ_G 0–8 MHz, pure-Lorentzian simulator/optimizer untouched) do NOT
  reproduce the half-μ attractor: μ̂/μ_true stays 0.99–1.03 with |Δμ|/σ_μ ≤ 0.12 at every
  σ_G. Gaussian broadening does not pull μ toward ½ — the attractor lives in the
  σ-channel / likelihood structure, not the line shape.

### 1.3 The structural fact that matters (this trial)
The real data files carry **(FWHM, fit-error) per scan — no photon counts**. The
optimizer's μ channel is a COUNT-based REINFORCE score: `(n_sim − μ)/σ_ref²` weighted by
the KDE responsibility `B = w.mean(0)`. On real data, μ information can therefore only
flow through the **σ_fit channel**: σ_fit is a function of n(μ) (fewer photons → larger
fit errors), so the KDE weight in the σ direction is the ONLY μ-sensitive link to the
data.

Working mechanism hypothesis for the ½-attractor (19a cell 1, explanation a):
real fit-errors are LARGE (1nW T20: err mean 3.5 MHz vs FWHM std 10.4 — checked
pre-run); the Lorentzian simulator at (μ_true, γ_true) produces smaller σ_fit, and the
only way to widen/fatten the simulated (FWHM, σ_fit) cloud is to lower μ (photon
starvation → worse fits → bigger errors). The count score then pins μ low, while the
(FWHM, σ_fit) KDE barely moves → flat μ likelihood, huge σ_μ.

### 1.4 Why a score redesign can help
The count score is a REINFORCE estimator of d log L/dμ through the count channel ONLY.
If the σ channel carries the real μ information, the optimizer is blind to it. Two cheap
fixes: (D1) calibrate σ_ref = σ_prop (principled standardized count residual — the fixed
σ_ref=10 was tuned on synthetic 17f), and (D2) add a z-form **σ-channel gradient** to the
μ score (dσ_sim/dμ = −σ_sim/2n_sim, fit-error ∝ 1/√n), mirroring the γ z-form score.
The autopsy (below) first maps the real μ landscape so the designs are interpretable.


## 2. PROPOSAL — autopsy + μ-score sweep (design, predictions, decision rule)

### 2.1 Design
- **Targets**: REAL scans (15-series load & filter), the 6 experiments of 19a's subset
  (1nW T20/T60/T100, 3nW T20/T60/T100) for direct comparability.
- **Optimizer**: 18c EXACTLY (z-form γ-score, anneal, clip, seeds). The ONLY change is
  the μ score, selected per design:
  - **D0 (control)**: 18c as-is, σ_ref=10 → expected: μ̂ stuck near init ½·μ_true, huge σ_μ.
  - **D1 (count-calibrated)**: σ_ref = σ_prop per experiment (standardized count residual).
  - **D2 (σ-channel gradient)**: D0 count term + z-form σ-channel term
    `Σ_i w_ji · (d_s/h_s) · (−σ_i/2n_i)/H_REF` in the μ score.
- **Autopsy first** (per exp, γ fixed at γ_true): NLL(μ) grid over
  μ/μ_true ∈ {0.3, 0.5, 0.7, 1.0, 1.3, 1.6}, with the count-channel and σ-channel score
  contributions at each μ, plus sim-vs-real σ_fit quantiles. → WHERE does the attractor
  come from, and is it a genuine likelihood minimum or just no-information + init?
- **Uncertainty**: Fisher/CRB at the fitted point with the DESIGN's own μ score
  (honest coverage of the actual estimator), raw γ-score for 15-series comparability,
  plus the spread diagnostic.

### 2.2 Predictions
- **Autopsy**: NLL(μ) is flat or has its minimum BELOW μ_true (≤ ½); the σ-channel score
  is the only one with a consistent μ pull; real σ_fit ≫ sim σ_fit at μ_true, closer at ½.
- **D0**: reproduces the 18-series μ̂ ≈ ½·μ_true with huge σ_μ.
- **D1**: changes the count-step scale but NOT the information content → μ̂ still flat/½.
- **D2**: if the σ channel carries real μ information, μ̂ moves toward μ_true with σ_μ
  honest (truth inside); if the real (FWHM, err) data genuinely cannot identify μ, all
  designs stay wide — the honest answer is then "μ unidentifiable from these features".

### 2.3 Decision rule → 19c
- **D2 recovers** μ̂ ≈ μ_true with |Δμ|/σ_μ ≲ 1.5 and σ_μ ≲ 0.5·μ_true → adopt the
  σ-channel score; 19c validates on all 14 exps + γ-side check.
- **All designs flat/wide** (σ_μ ≳ μ_true) → μ is fundamentally unidentifiable from
  (FWHM, err) alone; 19c reports honest-wide + designs a count-anchored experiment /
  external μ estimator, with the autopsy as evidence.
- **D1/D2 shift μ̂ but break coverage** → 19c tunes the hybrid (calibrated count +
  σ-channel weight) in closed loop.


In [ ]:
# ============================================================
# 3. CONFIG + BENCHMARK (series 19, trial b) — REAL data, μ-score designs
# ============================================================
import math, time, os, sys, json, warnings
warnings.filterwarnings('ignore')   # torch copy-construct warnings from src/samplers.py
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float32)

for _p in [os.getcwd(), os.path.join(os.getcwd(), '..'), os.path.join(os.getcwd(), '..', '..')]:
    if os.path.isdir(os.path.join(_p, 'src')):
        sys.path.insert(0, _p)
        REPO_ROOT = _p
        break
os.chdir(REPO_ROOT)

from src.fitting import nll, fwhm_from_theta, fit_profile
from src.samplers import draw_fixed_noise, sample_cauchy_truncated, FREQ_MIN, FREQ_MAX
from src.implicit import compute_fwhm_and_dgamma
from src.series19 import run_one, init_worker, parallel_map

# ---- run mode: SMOKE > PRE_SCREEN > FULL (env vars set by the driver) ----
SMOKE = os.environ.get('S19_SMOKE', '0') == '1'
PRE_SCREEN = os.environ.get('S19_PRESCREEN', '0') == '1'
if SMOKE:
    DESIGNS = ['D0']
    SWEEP_EXPS = ['1nW Trans60']
    N_RUNS, N_ITER = 6, 5
    TARGET_CAP, M_FINAL, N_AUTO = 40, 50, 40
elif PRE_SCREEN:
    DESIGNS = ['D0', 'D1', 'D2']
    SWEEP_EXPS = ['1nW Trans60', '3nW Trans60']
    N_RUNS, N_ITER = 40, 40
    TARGET_CAP, M_FINAL, N_AUTO = 400, 100, 100
else:
    DESIGNS = ['D0', 'D1', 'D2']
    SWEEP_EXPS = ['1nW Trans20', '1nW Trans60', '1nW Trans100',
                  '3nW Trans20', '3nW Trans60', '3nW Trans100']
    N_RUNS, N_ITER = 100, 100
    TARGET_CAP, M_FINAL, N_AUTO = None, 200, 200

# ---- fixed optimizer protocol (18c EXACTLY; only the μ score changes per design) ----
LR_MU, LR_GAMMA = 15.0, 0.5
SIGMA_REF, CLIP = 10.0, 10.0
GAMMA_ANNEAL, H_S_MIN = 0.5, 0.05
H_REF, LAMBDA_MEAN = 1.0, 0.0
SEED = 42
FISHER_BASE_SEED = 7000

EXPERIMENTS = [
    dict(name='1nW Trans20',  power='1nW', mu_true=17.316,  sigma_prop=4.141,  lam=2.286, gamma_true=8.5,  n_target=1138,
         data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans20.txt'),
    dict(name='1nW Trans60',  power='1nW', mu_true=61.374,  sigma_prop=9.851,  lam=2.593, gamma_true=8.5,  n_target=2424,
         data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans60.txt'),
    dict(name='1nW Trans100', power='1nW', mu_true=70.817,  sigma_prop=17.221, lam=2.636, gamma_true=8.5,  n_target=2455,
         data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans100.txt'),
    dict(name='3nW Trans20',  power='3nW', mu_true=34.279,  sigma_prop=8.319,  lam=2.264, gamma_true=14.1, n_target=2171,
         data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans20.txt'),
    dict(name='3nW Trans60',  power='3nW', mu_true=103.203, sigma_prop=23.95,  lam=2.741, gamma_true=14.1, n_target=2541,
         data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans60.txt'),
    dict(name='3nW Trans100', power='3nW', mu_true=175.707, sigma_prop=40.975, lam=3.087, gamma_true=14.1, n_target=2516,
         data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans100.txt'),
]

print(f'config: designs={DESIGNS}, exps={SWEEP_EXPS}, N_RUNS={N_RUNS}, N_ITER={N_ITER}, M_FINAL={M_FINAL}, N_AUTO={N_AUTO}')


In [ ]:
# ============================================================
# 4. REAL TARGETS (15-series load & filter) + machinery import
# ============================================================
def load_real_targets(exp):
    d = np.genfromtxt(exp['data_file'])
    fwhm_mhz = d[:, 0] * 1000.0
    err_mhz = d[:, 1] * 1000.0
    ok = ~np.isnan(fwhm_mhz) & ~np.isnan(err_mhz) & (fwhm_mhz > 0)
    filt = ok & (err_mhz / fwhm_mhz < 10.0)
    target_f = torch.tensor(fwhm_mhz[filt], dtype=torch.float32)
    target_s = torch.tensor(err_mhz[filt], dtype=torch.float32)
    if TARGET_CAP is not None and len(target_f) > TARGET_CAP:
        idx = np.random.default_rng(12345).choice(len(target_f), TARGET_CAP, replace=False)
        target_f = target_f[idx]
        target_s = target_s[idx]
    return target_f, target_s

def target_bandwidths(target_f, target_s):
    n = len(target_f)
    SCOTT = n ** (-1.0 / 6.0)
    H_F = float(target_f.std()) * SCOTT
    H_S = max(float(target_s.std()) * SCOTT, H_S_MIN)
    return H_F, H_S

def qs(x):
    return [float(np.quantile(x.detach().numpy() if torch.is_tensor(x) else x, q)) for q in (0.1, 0.25, 0.5, 0.75, 0.9)]

print('real-target loader ready')


In [ ]:
# ============================================================
# 5. LIKELIHOOD + μ-SCORE DESIGNS (D0/D1/D2) + AUTOPSY
# ============================================================
def kde_scores(sim_f, sim_s, sim_n, sim_df, sim_ds, data_f, data_s, h_f, h_s, mu, sigma_prop, gamma_scale=True):
    d_f = data_f[:, None] - sim_f[None, :]
    d_s = data_s[:, None] - sim_s[None, :]
    W = torch.exp(-0.5 * (d_f / h_f) ** 2 - 0.5 * (d_s / h_s) ** 2)
    w = W / W.sum(dim=1, keepdim=True).clamp_min(1e-12)
    score = (sim_n[None, :] - mu) / sigma_prop ** 2
    s_mu = (w * score).sum(dim=1)
    if gamma_scale:
        dlogG = ((d_f * sim_df[None, :]) / h_f + (d_s * sim_ds[None, :]) / h_s) / H_REF
    else:
        dlogG = (d_f * sim_df[None, :]) / h_f ** 2 + (d_s * sim_ds[None, :]) / h_s ** 2
    s_gamma = (w * dlogG).sum(dim=1)
    logp = torch.log((W.sum(dim=1) / len(sim_f)).clamp_min(1e-30))
    return s_mu, s_gamma, -logp.mean(), w

def grad_mu_design(design, w, sim_n, sim_s, data_s, h_s, mu, sigma_prop):
    '''Scalar μ gradient for the optimizer step (18c form for the count channel).'''
    B = w.mean(dim=0)
    if design == 'D0':
        score = (sim_n - mu) / SIGMA_REF ** 2
        return float(max(min(-(B - B.mean()) @ score, CLIP), -CLIP))
    if design == 'D1':
        score = (sim_n - mu) / sigma_prop ** 2
        return float(max(min(-(B - B.mean()) @ score, CLIP), -CLIP))
    if design == 'D2':
        score = (sim_n - mu) / SIGMA_REF ** 2
        grad_count = -(B - B.mean()) @ score
        d_s = data_s[:, None] - sim_s[None, :]
        dsigma_dmu = -sim_s / (2.0 * sim_n.clamp_min(1.0))   # fit error ~ 1/sqrt(n)
        s_sigma = (w * (d_s / h_s) * dsigma_dmu[None, :] / H_REF).sum(dim=1).mean()
        return float(max(min(-(grad_count + s_sigma), CLIP), -CLIP))
    raise ValueError(design)

def mu_score_point(design, w, sim_n, sim_s, data_s, h_s, mu, sigma_prop):
    '''Per-data-point μ score (Fisher / coverage of the actual estimator).'''
    if design == 'D0':
        score = (sim_n[None, :] - mu) / SIGMA_REF ** 2
        return (w * score).sum(dim=1)
    if design == 'D1':
        score = (sim_n[None, :] - mu) / sigma_prop ** 2
        return (w * score).sum(dim=1)
    if design == 'D2':
        score = (sim_n[None, :] - mu) / SIGMA_REF ** 2
        s_count = (w * score).sum(dim=1)
        d_s = data_s[:, None] - sim_s[None, :]
        dsigma_dmu = -sim_s[None, :] / (2.0 * sim_n[None, :].clamp_min(1.0))
        s_sigma = (w * (d_s / h_s) * dsigma_dmu / H_REF).sum(dim=1)
        return s_count + s_sigma
    raise ValueError(design)

def autopsy_exp(exp, target_f, target_s, pool):
    '''μ-likelihood autopsy at fixed γ=γ_true: NLL + score decomposition vs μ.'''
    H_F, H_S = target_bandwidths(target_f, target_s)
    mu_true, sigma_prop, lam, gamma_true = exp['mu_true'], exp['sigma_prop'], exp['lam'], exp['gamma_true']
    grid = np.array([0.3, 0.5, 0.7, 1.0, 1.3, 1.6]) * mu_true
    rows = []
    rng = np.random.default_rng(1234)
    for mu in grid:
        tasks, ns = [], []
        for _ in range(N_AUTO):
            u, b, n = draw_fixed_noise(mu, sigma_prop, lam, rng)
            tasks.append((gamma_true, u.numpy(), b.numpy()))
            ns.append(n)
        res = parallel_map(pool, tasks)
        ft = torch.tensor([r[0] for r in res], dtype=torch.float32)
        si_t = torch.tensor([r[1] for r in res], dtype=torch.float32)
        nt = torch.tensor(ns, dtype=torch.float32)
        dg_t = torch.tensor([r[2] for r in res], dtype=torch.float32)
        ds_t = torch.tensor([r[3] for r in res], dtype=torch.float32)
        s_mu, s_gamma, nll_val, w = kde_scores(ft, si_t, nt, dg_t, ds_t, target_f, target_s,
                                               H_F, H_S, mu, sigma_prop, gamma_scale=True)
        B = w.mean(dim=0)
        score = (nt - mu) / SIGMA_REF ** 2
        grad_count = float((B - B.mean()) @ score)      # d logL/dmu via count channel
        d_s = target_s[:, None] - si_t[None, :]
        dsigma_dmu = -si_t / (2.0 * nt.clamp_min(1.0))
        grad_sigma = float((w * (d_s / H_S) * dsigma_dmu[None, :] / H_REF).sum(dim=1).mean())
        rows.append(dict(mu=float(mu), mu_ratio=float(mu / mu_true), nll=float(nll_val),
                         grad_count=grad_count, grad_sigma=grad_sigma,
                         sim_s_q=qs(si_t), sim_f_mean=float(ft.mean())))
    return dict(exp=exp['name'], mu_true=mu_true, gamma_true=gamma_true,
                H_F=H_F, H_S=H_S, n_target=len(target_f),
                target_f_mean=float(target_f.mean()), target_s_mean=float(target_s.mean()),
                target_s_q=qs(target_s), rows=rows)

print('score designs + autopsy ready')


In [ ]:
# ============================================================
# 6. ONE EXPERIMENT: REAL targets + 18c optimizer, μ score per design
# ============================================================
def run_experiment(exp, design, target_f, target_s, pool):
    mu_true, sigma_prop = exp['mu_true'], exp['sigma_prop']
    lam, gamma_true = exp['lam'], exp['gamma_true']
    mu_init, gamma_init = 0.5 * mu_true, 0.5 * gamma_true
    H_F, H_S = target_bandwidths(target_f, target_s)

    mu_val, gamma_val = float(mu_init), float(gamma_init)
    history = []
    t0 = time.time()
    for step in range(N_ITER):
        rng2 = np.random.default_rng(SEED + step)
        tasks, ns = [], []
        for _ in range(N_RUNS):
            u, b, n = draw_fixed_noise(mu_val, sigma_prop, lam, rng2)
            tasks.append((gamma_val, u.numpy(), b.numpy()))
            ns.append(n)
        res = parallel_map(pool, tasks)
        ft = torch.tensor([r[0] for r in res], dtype=torch.float32)
        si_t = torch.tensor([r[1] for r in res], dtype=torch.float32)
        nt = torch.tensor(ns, dtype=torch.float32)
        dg_t = torch.tensor([r[2] for r in res], dtype=torch.float32)
        ds_t = torch.tensor([r[3] for r in res], dtype=torch.float32)

        s_mu, s_gamma, nll_val, w = kde_scores(ft, si_t, nt, dg_t, ds_t, target_f, target_s,
                                               H_F, H_S, mu_val, sigma_prop, gamma_scale=True)
        grad_mu = grad_mu_design(design, w, nt, si_t, target_s, H_S, mu_val, sigma_prop)
        grad_gamma = float(max(min(-s_gamma.mean(), CLIP), -CLIP))

        lr_mu_decay = LR_MU * (1.0 - step / N_ITER)
        mu_val -= lr_mu_decay * grad_mu
        mu_val = max(1.0, min(200.0, mu_val))
        gamma_val -= LR_GAMMA * (1.0 - GAMMA_ANNEAL * step / N_ITER) * grad_gamma
        gamma_val = max(0.1, min(100.0, gamma_val))
        history.append(dict(step=step, mu=mu_val, gamma=gamma_val, nll=float(nll_val)))

    return dict(exp=exp['name'], power=exp['power'], design=design,
                mu_true=mu_true, gamma_true=gamma_true, sigma_prop=sigma_prop, lam=lam,
                mu_init=mu_init, gamma_init=gamma_init, mu_final=mu_val, gamma_final=gamma_val,
                nll_final=float(nll_val), t_elapsed=time.time() - t0,
                target_f=target_f, target_s=target_s, H_F=H_F, H_S=H_S, history=history)

print('run_experiment ready')


In [ ]:
# ============================================================
# 7. FISHER / CRB at the fitted point (design's own μ score for honest
#    coverage; raw γ-score for 15-series comparability) + spread diagnostic
# ============================================================
def fisher_at_fitted(r, pool):
    mu_f, gamma_f = r['mu_final'], r['gamma_final']
    sp, lam = r['sigma_prop'], r['lam']
    design = r['design']
    rng = np.random.default_rng(FISHER_BASE_SEED)
    tasks, ns = [], []
    for _ in range(M_FINAL):
        u, b, n = draw_fixed_noise(mu_f, sp, lam, rng)
        tasks.append((gamma_f, u.numpy(), b.numpy()))
        ns.append(n)
    res = parallel_map(pool, tasks)
    ft = torch.tensor([x[0] for x in res], dtype=torch.float32)
    si_t = torch.tensor([x[1] for x in res], dtype=torch.float32)
    nt = torch.tensor(ns, dtype=torch.float32)
    dg_t = torch.tensor([x[2] for x in res], dtype=torch.float32)
    ds_t = torch.tensor([x[3] for x in res], dtype=torch.float32)
    target_f, target_s = r['target_f'], r['target_s']

    _, s_gamma, _, w = kde_scores(ft, si_t, nt, dg_t, ds_t, target_f, target_s,
                                  r['H_F'], r['H_S'], mu_f, sp, gamma_scale=False)
    s_mu = mu_score_point(design, w, nt, si_t, target_s, r['H_S'], mu_f, sp)
    s = torch.stack([s_mu, s_gamma], dim=1)
    J = s.T @ s / len(target_f)
    Jinv = torch.linalg.inv(J + 1e-8 * torch.eye(2))
    std_mu = float(math.sqrt(Jinv[0, 0]))
    std_gamma = float(math.sqrt(Jinv[1, 1]))
    corr = float(Jinv[0, 1] / torch.sqrt(Jinv[0, 0] * Jinv[1, 1]))
    spread_ratio = float(ft.std()) / float(target_f.std())
    return dict(exp=r['exp'], design=design, mu_f=mu_f, gamma_f=gamma_f,
                mu_true=r['mu_true'], gamma_true=r['gamma_true'], n_target=len(target_f),
                std_mu=std_mu, std_gamma=std_gamma, corr=corr, spread_ratio=spread_ratio,
                dmu_sigma=abs(mu_f - r['mu_true']) / std_mu,
                dgamma_sigma=abs(gamma_f - r['gamma_true']) / std_gamma)

print('fisher_at_fitted ready')


In [ ]:
# ============================================================
# 8. RUN: autopsy first, then 3 designs x 6 experiments, then Fisher
# ============================================================
import multiprocessing as _mp
from concurrent.futures import ProcessPoolExecutor as _PPE

N_CPUS = os.cpu_count() or 4
N_WORKERS = 4

t_all = time.time()

# ---- 8a. AUTOPSY (μ landscape at fixed γ_true) ----
print('AUTOPSY ...', flush=True)
autopsy = []
with _PPE(max_workers=N_WORKERS, mp_context=_mp.get_context('fork'), initializer=init_worker) as pool:
    for exp in EXPERIMENTS:
        if exp['name'] not in SWEEP_EXPS:
            continue
        tf, ts = load_real_targets(exp)
        a = autopsy_exp(exp, tf, ts, pool)
        autopsy.append(a)
        best = min(a['rows'], key=lambda r: r['nll'])
        print(f"  {exp['name']:>12}: NLL min at mu/mu_true={best['mu_ratio']:.2f} | "
              f"real err {a['target_s_mean']:.2f} vs sim@1.0 {a['rows'][3]['sim_s_q'][2]:.2f} "
              f"vs sim@0.5 {a['rows'][1]['sim_s_q'][2]:.2f}", flush=True)
print(f'autopsy done in {(time.time() - t_all) / 60:.1f} min', flush=True)

# ---- 8b. SWEEP (designs x exps) ----
results = []
with _PPE(max_workers=N_WORKERS, mp_context=_mp.get_context('fork'), initializer=init_worker) as pool:
    for design in DESIGNS:
        for exp in EXPERIMENTS:
            if exp['name'] not in SWEEP_EXPS:
                continue
            tf, ts = load_real_targets(exp)
            print(f"Running {design} {exp['name']:>12} ...", end=' ', flush=True)
            r = run_experiment(exp, design, tf, ts, pool)
            results.append(r)
            print(f"mu {r['mu_true']:.2f} -> {r['mu_final']:.2f} | "
                  f"gamma {r['gamma_true']:.1f} -> {r['gamma_final']:.2f} | "
                  f"NLL {r['nll_final']:.2f} | {r['t_elapsed'] / 60:.1f} min", flush=True)

# ---- 8c. FISHER at each fitted point ----
fisher = []
with _PPE(max_workers=N_WORKERS, mp_context=_mp.get_context('fork'), initializer=init_worker) as pool:
    for r in results:
        f = fisher_at_fitted(r, pool)
        fisher.append(f)
        print(f"FISHER {f['design']} {f['exp']:>12}: |dmu|/sig={f['dmu_sigma']:.2f} "
              f"sig_mu={f['std_mu']:.2f} | |dgamma|/sig={f['dgamma_sigma']:.2f} "
              f"sig_gamma={f['std_gamma']:.2f}", flush=True)

out = dict(mode='FULL', designs=DESIGNS, exps=SWEEP_EXPS, n_runs=N_RUNS, n_iter=N_ITER,
           m_final=M_FINAL, rows=fisher)
json.dump(out, open('/tmp/19b_sweep.json', 'w'), indent=1)
json.dump(autopsy, open('/tmp/19b_autopsy.json', 'w'), indent=1)
print(f'\nTotal: {(time.time() - t_all) / 60:.1f} min — saved /tmp/19b_sweep.json + /tmp/19b_autopsy.json', flush=True)


In [ ]:
# ============================================================
# 9. FIGURES -> /tmp/19b_figs/
# ============================================================
os.makedirs('/tmp/19b_figs', exist_ok=True)
sweep = json.load(open('/tmp/19b_sweep.json'))
autopsy = json.load(open('/tmp/19b_autopsy.json'))
rows = sweep['rows']
EXPS = sweep['exps']
DESIGNS_S = sweep['designs']
DCOL = {'D0': '#888888', 'D1': '#1f77b4', 'D2': '#d62728'}

# ---- fig1: autopsy NLL + score decomposition ----
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, a in zip(axes.ravel(), autopsy):
    rr = a['rows']
    mr = [r['mu_ratio'] for r in rr]
    ax.plot(mr, [r['nll'] for r in rr], 'k-o', label='NLL (left)')
    ax.set_xlabel('mu / mu_true')
    ax.set_ylabel('NLL', color='k')
    ax2 = ax.twinx()
    ax2.plot(mr, [r['grad_count'] for r in rr], 'b--s', label='count score (right)')
    ax2.plot(mr, [r['grad_sigma'] for r in rr], 'r--^', label='sigma score (right)')
    ax2.axhline(0, color='gray', lw=0.5)
    ax2.set_ylabel('score', color='gray')
    ax.axvline(1.0, color='g', lw=0.8, ls=':')
    ax.axvline(0.5, color='orange', lw=0.8, ls=':')
    ax.set_title(a['exp'])
    ax.grid(alpha=0.3)
fig.suptitle('19b autopsy: mu-likelihood at fixed gamma_true (green=truth, orange=1/2)')
fig.tight_layout()
fig.savefig('/tmp/19b_figs/fig1_autopsy.png', dpi=110)
plt.close(fig)

# ---- fig2: sim vs real sigma_fit quantiles ----
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, a in zip(axes.ravel(), autopsy):
    rr = a['rows']
    mr = [r['mu_ratio'] for r in rr]
    med = [r['sim_s_q'][2] for r in rr]
    lo = [r['sim_s_q'][1] for r in rr]
    hi = [r['sim_s_q'][3] for r in rr]
    ax.plot(mr, med, 'b-o', label='sim sigma_fit median')
    ax.fill_between(mr, lo, hi, color='b', alpha=0.15, label='sim IQR')
    tq = a['target_s_q']
    ax.axhspan(tq[1], tq[3], color='r', alpha=0.12)
    ax.axhline(tq[2], color='r', ls='--', label='real err median')
    ax.axvline(1.0, color='g', lw=0.8, ls=':')
    ax.axvline(0.5, color='orange', lw=0.8, ls=':')
    ax.set_xlabel('mu / mu_true')
    ax.set_ylabel('sigma_fit (MHz)')
    ax.set_title(a['exp'])
    ax.grid(alpha=0.3)
fig.suptitle('19b sigma-channel: simulated fit-error vs REAL fit-error (red band = real IQR)')
fig.tight_layout()
fig.savefig('/tmp/19b_figs/fig2_sigma_channel.png', dpi=110)
plt.close(fig)

# ---- fig3: mu recovery by design ----
fig, ax = plt.subplots(figsize=(12, 5.5))
x = np.arange(len(EXPS))
wdt = 0.25
for k, d in enumerate(DESIGNS_S):
    ys, errs = [], []
    for e in EXPS:
        r = next(r for r in rows if r['design'] == d and r['exp'] == e)
        ys.append(r['mu_f'] / r['mu_true'])
        errs.append(r['std_mu'] / r['mu_true'])
    ax.errorbar(x + (k - 1) * wdt, ys, yerr=errs, fmt='o', color=DCOL[d],
                label=d, capsize=3)
ax.axhline(1.0, color='g', ls=':', label='truth')
ax.axhline(0.5, color='orange', ls=':', label='1/2 attractor / init')
ax.set_xticks(x)
ax.set_xticklabels([e.replace('nW ', 'nW\n') for e in EXPS])
ax.set_ylabel('mu_hat / mu_true')
ax.set_title('19b mu recovery by design (error bars = Fisher sigma_mu)')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('/tmp/19b_figs/fig3_mu_recovery.png', dpi=110)
plt.close(fig)

# ---- fig4: coverage map |d|/sigma ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, key, title in zip(axes, ['dmu_sigma', 'dgamma_sigma'], ['|dmu|/sigma_mu', '|dgamma|/sigma_gamma']):
    M = np.zeros((len(DESIGNS_S), len(EXPS)))
    for i, d in enumerate(DESIGNS_S):
        for j, e in enumerate(EXPS):
            r = next(r for r in rows if r['design'] == d and r['exp'] == e)
            M[i, j] = r[key]
    im = ax.imshow(M, cmap='RdYlGn_r', vmin=0, vmax=4)
    ax.set_xticks(range(len(EXPS)))
    ax.set_xticklabels([e.replace('nW ', 'nW\n') for e in EXPS], rotation=45, ha='right')
    ax.set_yticks(range(len(DESIGNS_S)))
    ax.set_yticklabels(DESIGNS_S)
    for i in range(len(DESIGNS_S)):
        for j in range(len(EXPS)):
            v = M[i, j]
            ax.text(j, i, f'{v:.1f}', ha='center', va='center',
                    color='white' if v > 2 else 'black', fontsize=9)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label='|dev|/sigma  (>2 = tight-wrong)')
fig.suptitle('19b coverage map (rows=design, cols=exp)')
fig.tight_layout()
fig.savefig('/tmp/19b_figs/fig4_coverage.png', dpi=110)
plt.close(fig)

# ---- fig5: mu paths for 1nW Trans60 ----
fig, ax = plt.subplots(figsize=(10, 5))
for r in results:
    if r['exp'] != '1nW Trans60':
        continue
    h = r['history']
    ax.plot([s['step'] for s in h], [s['mu'] for s in h], color=DCOL[r['design']],
            label=f"{r['design']} -> {r['mu_final']:.1f}")
ax.axhline(61.374, color='g', ls=':', label='truth 61.4')
ax.axhline(30.7, color='orange', ls=':', label='1/2 init 30.7')
ax.set_xlabel('step')
ax.set_ylabel('mu')
ax.set_title('19b mu paths — 1nW Trans60 (real targets)')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('/tmp/19b_figs/fig5_paths.png', dpi=110)
plt.close(fig)

print('figures saved to /tmp/19b_figs/')


## 10. CONCLUSION ANALYSIS (filled after the run)

*— to be written after the sweep completes —*
